# Ludwig 2 · T0.2 — Velocidad y memoria en 2×T4

Mide con el **modelo real** (1.96B totales / 0.49B activos), FSDP2 entre las dos T4, FP16 con escalado de pérdida, Muon (momento bf16) + AdamW, pérdida por bloques y balanceo. Solo los datos son aleatorios.

| Bloque | Qué decide |
|---|---|
| A | El kernel de MoE más rápido en la T4: `eager`, `batched_mm` o `grouped_mm` |
| B | Si un micro-batch más grande sube los tokens/s sin pasarse de memoria |
| C–D | Cuánto cuestan 4K y 16K de contexto, y si 16K cabe |
| E | El plan B de 32 expertos |
| F | El costo de entrenar el MTP |
| G | Guardado y carga del checkpoint distribuido: tiempo, tamaño y restauración exacta |

**Kaggle:** acelerador *GPU T4 x2*, Internet *On*. Tarda ~45–75 min. Cada medición escribe una línea en `results/t0_2.jsonl`, así que si la sesión se corta no se pierde lo ya medido.

In [ ]:
REPO_URL = "https://github.com/FelipeGarciaRpo/ludwig-lm.git"   # o "" si subiste el repo como dataset

In [ ]:
import os, subprocess, sys, json
os.chdir("/kaggle/working")
if not os.path.exists("ludwig-lm"):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ludwig-lm"], check=True)
    else:  # el repo subido como dataset de Kaggle: se busca la carpeta que contiene ludwig_lm/
        import glob
        src = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/ludwig_lm/__init__.py", recursive=True)[0]))
        subprocess.run(["cp", "-r", src, "ludwig-lm"], check=True)
os.chdir("/kaggle/working/ludwig-lm")
!pip install -q "transformers==5.22.2" safetensors sentencepiece protobuf
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CPUs", os.cpu_count(),
      "| GPUs", torch.cuda.device_count())
!nvidia-smi --query-gpu=name,memory.total,power.limit --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
import threading, time
RESULTS = "results/t0_2.jsonl"
KEEP = ("[model]", "  step", "[result]", "[ckpt]", "Error", "OutOfMemory", "out of memory", "Traceback")

def run(tag, timeout_min=25, **kw):
    """Lanza una medición con torchrun en las 2 GPUs, muestra el progreso y devuelve su fila de resultados."""
    args = " ".join(f"--{k.replace('_', '-')}" + ("" if v is True else f" {v}") for k, v in kw.items())
    cmd = (f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True torchrun --nproc_per_node 2 "
           f"scripts/bench_train.py --tag {tag} --out {RESULTS} {args}")
    print(">>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         start_new_session=True)
    killed = []
    def kill():  # torchrun runs each worker in its own session: kill by command line, TERM first, then KILL
        killed.append(True)
        subprocess.run(["pkill", "-TERM", "-f", "scripts/bench_train.py"])
        time.sleep(20)
        subprocess.run(["pkill", "-KILL", "-f", "scripts/bench_train.py"])
    timer = threading.Timer(timeout_min * 60, kill)
    timer.start()
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-30:]
        if any(k in line for k in KEEP):
            print(line, end="", flush=True)
    p.wait()
    timer.cancel()
    if p.returncode != 0:
        print(f"[{tag}] terminó con código {p.returncode}; últimas líneas:\n" + "".join(tail[-12:]))
    rows = [json.loads(l) for l in open(RESULTS)] if os.path.exists(RESULTS) else []
    row = next((r for r in reversed(rows) if r["tag"] == tag), None)
    if row is None:  # se cayó o se pasó del tiempo antes de escribir su fila: se anota igual para el resumen
        row = {"tag": tag, "status": "timeout" if killed else f"crash (código {p.returncode})", **kw}
        with open(RESULTS, "a") as f:
            f.write(json.dumps(row) + "\n")
    return row

## A. Kernel de MoE (2K de contexto, micro-batch 4)

En la T4 (SM75), `grouped_mm` no tiene el kernel nativo de PyTorch (pide SM80+): transformers usa su respaldo, que ordena los tokens por experto y hace una multiplicación por experto. `batched_mm` y `eager` son las otras dos formas.

In [ ]:
res = {}
for impl in ["grouped_mm", "batched_mm", "eager"]:
    res[impl] = run(f"A-{impl}", moe_impl=impl, seq=2048, micro=4, accum=2, steps=4)
ok = {k: v for k, v in res.items() if v and v["status"] == "ok"}
BEST = max(ok, key=lambda k: ok[k]["tokens_per_s"]) if ok else "batched_mm"
print("Kernel más rápido:", BEST)

## B–F. Micro-batch, contexto, plan B y MTP

In [ ]:
run("B-micro8", moe_impl=BEST, seq=2048, micro=8, accum=1, steps=4)
run("C-seq4k", moe_impl=BEST, seq=4096, micro=2, accum=2, steps=4)
run("D-seq16k", moe_impl=BEST, seq=16384, micro=1, accum=1, steps=3)
run("E-32experts", moe_impl=BEST, experts=32, seq=2048, micro=4, accum=2, steps=4)
run("F-mtp", moe_impl=BEST, seq=2048, micro=4, accum=2, steps=4, mtp=1.0)

## G. Checkpoint distribuido: guardar y cargar

In [ ]:
# ~12 GB: va a /tmp, no a /kaggle/working (que guarda la salida del notebook y tiene tope de ~20 GB)
run("G-ckpt", moe_impl=BEST, seq=2048, micro=4, accum=1, steps=3, ckpt_test=True, ckpt_dir="/tmp/ludwig_ckpt_test")
!rm -rf /tmp/ludwig_ckpt_test

## Resumen y recomendación

In [ ]:
import pandas as pd
df = pd.DataFrame([json.loads(l) for l in open(RESULTS)])
cols = ["tag", "status", "experts", "seq", "micro", "accum", "moe_impl", "tokens_per_s", "step_s", "peak_reserved_GiB",
        "hours_for_25B", "loss_scale_skips", "load_max_over_mean", "ckpt_save_s", "ckpt_GiB", "ckpt_restore_exact"]
display(df[[c for c in cols if c in df.columns]])
ok40 = df[(df.status == "ok") & (df.experts == 40) & (df.seq == 2048)]
fits = ok40[ok40.peak_reserved_GiB <= 13.5]
if len(fits):
    b = fits.sort_values("tokens_per_s", ascending=False).iloc[0]
    print(f"Mejor configuración a 2K que cabe con margen: {b.tag} -> {b.tokens_per_s:,.0f} tokens/s, "
          f"pico {b.peak_reserved_GiB} GiB, ~{b.hours_for_25B:,.0f} h para 25B tokens")
elif len(ok40):
    print("Ninguna configuración de 40 expertos quedó bajo 13.5 GiB: mirar el plan B (fila E-32experts).")
else:
    print("Ninguna medición de 40 expertos a 2K terminó bien: revisar los errores de arriba.")

## Qué hacer con el resultado

Descarga `results/t0_2.jsonl` y súbelo al repo. Con esos números se fijan el kernel de MoE, el micro-batch, si 16K cabe y si van 40 o 32 expertos. Después viene la compuerta G0: una prueba sostenida de ≥6 h con datos reales, con guardado y reanudación en otra cuenta.